# Seq2seq から Transformer へ：なぜ革命が必要だったのか

このノートブックでは、Transformer が登場する前の機械翻訳の仕組み **Seq2seq** と、
その問題点を理解し、なぜ Transformer が必要だったのかを学びます。

## 目次
1. Seq2seq とは何か？
2. RNN（再帰型ニューラルネットワーク）の仕組み
3. Seq2seq の問題点①：ボトルネック問題
4. Seq2seq の問題点②：逐次処理の遅さ
5. Transformer のアテンション機構による解決
6. Q, K, V：アテンションの3つの役者
7. アテンションの計算式
8. Transformer ブロックの構造
9. コードで比較：RNN vs アテンション
10. まとめ

## 1. Seq2seq とは何か？

**Seq2seq（Sequence-to-Sequence）** は 2014 年に登場した、ある系列（シーケンス）を別の系列に変換する深層学習モデルです。

| 項目 | 内容 |
|------|------|
| 登場年 | 2014年 |
| 開発者 | Google（Sutskever et al.） |
| 用途 | 機械翻訳、要約、対話生成など |
| 構造 | エンコーダ＋デコーダ（RNNベース） |

### 2016年：Google翻訳に採用

Seq2seq の威力が認められ、2016年には Google 翻訳に採用されました。
それまでの統計的機械翻訳（SMT）から、ニューラル機械翻訳（NMT）への転換点となりました。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 日本語フォント設定
plt.rcParams['font.family'] = ['DejaVu Sans', 'Hiragino Sans', 'Yu Gothic', 'Meiryo', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 機械翻訳の歴史タイムラインを可視化
fig, ax = plt.subplots(figsize=(14, 4))

# タイムライン
years = [2014, 2016, 2017, 2018, 2022, 2023]
events = [
    'Seq2seq\n登場',
    'Google翻訳に\nSeq2seq採用',
    'Transformer\n登場',
    'BERT\n登場',
    'ChatGPT\n登場',
    'GPT-4\n登場'
]
colors = ['#BBDEFB', '#90CAF9', '#FF8A65', '#FFE082', '#A5D6A7', '#81C784']

ax.set_xlim(2012, 2025)
ax.set_ylim(-1, 2)
ax.axis('off')

# 横線
ax.plot([2013, 2024], [0, 0], 'k-', lw=2)

for year, event, color in zip(years, events, colors):
    # 年のマーカー
    ax.plot(year, 0, 'ko', markersize=10)
    ax.text(year, -0.3, str(year), ha='center', fontsize=11, fontweight='bold')
    
    # イベントボックス
    rect = mpatches.FancyBboxPatch((year-0.8, 0.4), 1.6, 1.2,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(year, 1.0, event, ha='center', va='center', fontsize=9, fontweight='bold')
    
    # 矢印
    ax.annotate('', xy=(year, 0.1), xytext=(year, 0.35),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# 強調
ax.annotate('RNN時代', xy=(2015, -0.7), fontsize=10, color='#1565C0', fontweight='bold', ha='center')
ax.annotate('Transformer時代', xy=(2020, -0.7), fontsize=10, color='#E65100', fontweight='bold', ha='center')
ax.plot([2016.5, 2016.5], [-0.9, 1.8], 'r--', lw=2, alpha=0.5)

ax.set_title('機械翻訳・LLM の歴史', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("ポイント:")
print("  2014年: Seq2seq（RNNベース）が機械翻訳を革新")
print("  2017年: Transformer がさらに大きな革命を起こす")
print("  → なぜ Seq2seq では不十分だったのか？これを理解することが重要！")

## 2. RNN（再帰型ニューラルネットワーク）の仕組み

Seq2seq の中核は **RNN（Recurrent Neural Network：再帰型ニューラルネットワーク）** です。

### RNN のポイント

| 特徴 | 説明 |
|------|------|
| 逐次処理 | 入力を **1つずつ順番に** 処理する |
| 隠れ状態 | 各ステップで「これまでの情報」を **隠れ状態（hidden state）** として保持 |
| 情報の受け渡し | 前のステップの隠れ状態を、次のステップに渡す |

### 数式で見る RNN

$$h_t = f(h_{t-1}, x_t)$$

- $x_t$：時刻 $t$ の入力（例：単語「How」）
- $h_{t-1}$：前の時刻の隠れ状態（これまでの文脈を圧縮したもの）
- $h_t$：現在の隠れ状態（更新後）
- $f$：活性化関数を含む変換（tanh など）

**要するに**：RNN は「前までの情報 + 今の入力」から「新しい情報」を作る処理を繰り返します。

In [ ]:
# RNN の逐次処理を可視化
fig, ax = plt.subplots(figsize=(14, 5))

words = ['How', 'are', 'you', '?']
n_words = len(words)

ax.set_xlim(-1, 14)
ax.set_ylim(-1, 4)
ax.axis('off')
ax.set_title('RNN の逐次処理：単語を1つずつ処理して隠れ状態を更新', fontsize=13, fontweight='bold')

for i, word in enumerate(words):
    x = i * 3.2 + 1
    
    # 入力単語
    ax.text(x, 0, f'[{word}]', ha='center', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='#E3F2FD', edgecolor='#1565C0'))
    
    # 入力矢印
    ax.annotate('', xy=(x, 1.3), xytext=(x, 0.4),
                arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))
    ax.text(x + 0.3, 0.85, f'$x_{i+1}$', fontsize=10, color='#1565C0')
    
    # RNN セル
    rect = mpatches.FancyBboxPatch((x-0.6, 1.4), 1.2, 1.0,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#FFCC80', edgecolor='#E65100', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, 1.9, 'RNN', ha='center', fontsize=11, fontweight='bold')
    
    # 隠れ状態の矢印（次のRNNへ）
    if i < n_words - 1:
        ax.annotate('', xy=(x + 2.0, 1.9), xytext=(x + 0.7, 1.9),
                    arrowprops=dict(arrowstyle='->', color='#E65100', lw=2))
        ax.text(x + 1.35, 2.2, f'$h_{i+1}$', fontsize=10, color='#E65100')
    
    # 出力矢印
    ax.annotate('', xy=(x, 3.0), xytext=(x, 2.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    ax.text(x, 3.2, f'$h_{i+1}$', ha='center', fontsize=10, color='gray')

# 初期隠れ状態
ax.text(-0.3, 1.9, '$h_0$', fontsize=10, color='#E65100',
        bbox=dict(boxstyle='round', facecolor='#FFF3E0', edgecolor='#E65100'))
ax.annotate('', xy=(0.3, 1.9), xytext=(0.1, 1.9),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2))

# 最終隠れ状態を強調
final_x = (n_words - 1) * 3.2 + 1
ax.text(final_x + 1.5, 1.9, f'$h_{n_words}$\n(最終隠れ状態)', ha='center', fontsize=11,
        fontweight='bold', color='#E65100',
        bbox=dict(boxstyle='round', facecolor='#FFE0B2', edgecolor='#E65100', linewidth=2))
ax.annotate('', xy=(final_x + 1.0, 1.9), xytext=(final_x + 0.7, 1.9),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2))

plt.tight_layout()
plt.show()

print("RNN の特徴:")
print("  1. 単語を1つずつ順番に処理する（並列処理できない）")
print("  2. 各ステップで隠れ状態 h を更新する")
print("  3. 最終的な隠れ状態 h₄ に「文全体の情報」が圧縮される")

In [ ]:
# RNN の計算を numpy で実装してみる
np.random.seed(42)

# パラメータ設定
input_dim = 4    # 入力ベクトルの次元（単語埋め込みの次元）
hidden_dim = 3   # 隠れ状態の次元

# 単語の埋め込みベクトル（簡略化のためランダム値）
word_embeddings = {
    'How': np.array([0.5, 0.2, 0.1, 0.8]),
    'are': np.array([0.3, 0.7, 0.4, 0.2]),
    'you': np.array([0.6, 0.1, 0.9, 0.3]),
    '?':   np.array([0.1, 0.4, 0.2, 0.5]),
}

# RNN の重み行列（ランダム初期化）
W_h = np.random.randn(hidden_dim, hidden_dim) * 0.5  # 隠れ状態用
W_x = np.random.randn(hidden_dim, input_dim) * 0.5   # 入力用
b = np.zeros(hidden_dim)                              # バイアス

def rnn_step(h_prev, x):
    """RNN の1ステップを計算
    
    h_t = tanh(W_h @ h_{t-1} + W_x @ x_t + b)
    """
    return np.tanh(W_h @ h_prev + W_x @ x + b)

# 逐次処理を実行
print("=== RNN の逐次処理を追跡 ===")
print(f"隠れ状態の次元: {hidden_dim}")
print()

h = np.zeros(hidden_dim)  # 初期隠れ状態（ゼロベクトル）
print(f"初期状態 h₀ = {h}")
print()

for i, word in enumerate(['How', 'are', 'you', '?']):
    x = word_embeddings[word]
    h_new = rnn_step(h, x)
    
    print(f"ステップ {i+1}: 入力 = [{word}]")
    print(f"  x_{i+1} = {x}")
    print(f"  h_{i} (前の隠れ状態) = [{h[0]:.3f}, {h[1]:.3f}, {h[2]:.3f}]")
    print(f"  h_{i+1} (新しい隠れ状態) = [{h_new[0]:.3f}, {h_new[1]:.3f}, {h_new[2]:.3f}]")
    print()
    
    h = h_new

print(f"最終隠れ状態 h₄ = [{h[0]:.3f}, {h[1]:.3f}, {h[2]:.3f}]")
print()
print("→ この h₄ に『How are you ?』の文全体の情報が圧縮されている（はず）")

## 3. Seq2seq の問題点①：ボトルネック問題

Seq2seq では、エンコーダの **最終隠れ状態だけ** をデコーダに渡します。

### 何が問題か？

入力文がどんなに長くても、**固定長のベクトル1つ** に全情報を詰め込まなければなりません。

```
入力: "The quick brown fox jumps over the lazy dog near the river bank."
           ↓ エンコーダで処理
最終隠れ状態: [0.23, -0.15, 0.78, ...]  ← たった1つのベクトル！
           ↓ デコーダに渡す
出力: 「素早い茶色のキツネが...」
```

### 情報のボトルネック

| 入力文の長さ | 最終隠れ状態のサイズ | 問題 |
|-------------|-------------------|------|
| 5単語 | 256次元（固定） | 余裕あり |
| 20単語 | 256次元（固定） | ちょっとキツい |
| 100単語 | 256次元（固定） | **情報が失われる！** |

これを **ボトルネック問題（Bottleneck Problem）** と呼びます。

In [ ]:
# ボトルネック問題を可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# === 左: Seq2seq のボトルネック ===
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('Seq2seq のボトルネック問題', fontsize=13, fontweight='bold', color='#D32F2F')

# エンコーダ（入力が多い）
encoder_words = ['How', 'are', 'you', '?']
for i, word in enumerate(encoder_words):
    ax.text(0.5 + i * 1.0, 4.5, f'[{word}]', ha='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='#E3F2FD', edgecolor='#1565C0'))
    ax.annotate('', xy=(2, 3.5), xytext=(0.5 + i * 1.0, 4.2),
                arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1, alpha=0.5))

# ボトルネック（1つのベクトル）
bottleneck = mpatches.FancyBboxPatch((1.2, 2.5), 1.6, 0.8,
                                      boxstyle='round,pad=0.1',
                                      facecolor='#FFCDD2', edgecolor='#D32F2F', linewidth=3)
ax.add_patch(bottleneck)
ax.text(2, 2.9, '最終隠れ状態\n(固定長)', ha='center', fontsize=9, fontweight='bold', color='#D32F2F')

# デコーダ（出力が多い）
decoder_words = ['Como', 'estas', '?']
for i, word in enumerate(decoder_words):
    ax.annotate('', xy=(5 + i * 1.2, 1.5), xytext=(2.8, 2.9),
                arrowprops=dict(arrowstyle='->', color='#E65100', lw=1, alpha=0.5))
    ax.text(5 + i * 1.2, 1.0, f'[{word}]', ha='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='#FFE0B2', edgecolor='#E65100'))

# 問題点の説明
ax.text(5, 4.5, '問題：\n文がどんなに長くても\n最終隠れ状態は\n固定サイズ！', ha='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='#FFEBEE', edgecolor='#D32F2F'),
        color='#D32F2F')

# === 右: 文の長さと情報損失の関係 ===
ax = axes[1]

# 文の長さと翻訳精度の関係（概念的なグラフ）
sentence_lengths = np.array([5, 10, 15, 20, 25, 30, 35, 40])
seq2seq_accuracy = 100 * np.exp(-0.02 * (sentence_lengths - 5))  # 長くなると急激に低下
transformer_accuracy = 100 - 0.5 * (sentence_lengths - 5)  # ゆるやかに低下

ax.plot(sentence_lengths, seq2seq_accuracy, 'o-', color='#D32F2F', lw=2, markersize=8, label='Seq2seq（RNN）')
ax.plot(sentence_lengths, transformer_accuracy, 's-', color='#4CAF50', lw=2, markersize=8, label='Transformer')

ax.set_xlabel('入力文の長さ（単語数）', fontsize=11)
ax.set_ylabel('翻訳精度（相対値）', fontsize=11)
ax.set_title('文の長さと翻訳精度の関係（概念図）', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(60, 105)

# 注釈
ax.annotate('長い文で\n精度が急落', xy=(30, 78), fontsize=10, color='#D32F2F',
            ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("ボトルネック問題:")
print("  Seq2seq は入力文全体の情報を『1つの固定長ベクトル』に押し込める")
print("  → 長い文では情報が失われ、翻訳精度が大きく低下する")
print("  → これが Transformer 登場の大きな動機の1つ")

In [ ]:
# ボトルネック問題を数値で実感する

def calculate_information_density(sentence_length, hidden_dim):
    """1単語あたりの情報量（次元数）を計算"""
    return hidden_dim / sentence_length

hidden_dim = 256  # 典型的な隠れ状態の次元

print("=== 1単語あたりの情報量 ===")
print(f"隠れ状態の次元: {hidden_dim}")
print()
print(f"{'文の長さ':>10s} {'1単語あたりの次元':>18s} {'状況':>15s}")
print("-" * 50)

for length in [5, 10, 20, 50, 100, 200]:
    info_per_word = calculate_information_density(length, hidden_dim)
    if info_per_word >= 20:
        status = "余裕あり"
    elif info_per_word >= 5:
        status = "ギリギリ"
    elif info_per_word >= 2:
        status = "情報不足"
    else:
        status = "深刻な情報損失"
    
    print(f"{length:>8d}語 {info_per_word:>16.1f}次元 {status:>15s}")

print()
print("→ 長い文ほど、1単語に割り当てられる情報量が減る")
print("→ これがボトルネック問題の本質")

## 4. Seq2seq の問題点②：逐次処理の遅さ

RNN のもう1つの問題は、**逐次処理（Sequential Processing）** が必須であることです。

### 並列処理ができない

RNN では、$h_t$ を計算するために $h_{t-1}$ が必要です。

```
h₁ = f(h₀, x₁)  ← まず x₁ を処理
h₂ = f(h₁, x₂)  ← h₁ がないと計算できない！
h₃ = f(h₂, x₃)  ← h₂ がないと計算できない！
h₄ = f(h₃, x₄)  ← h₃ がないと計算できない！
```

つまり、単語数が N 個あると、**N ステップの計算が順番に** 必要です。

### GPU が活かせない

| 処理方式 | GPU の得意/不得意 |
|----------|------------------|
| 並列処理 | GPU は**大量の並列計算**が得意 |
| 逐次処理 | 待ち時間が発生し、GPU を活かせない |

長い文を処理するとき、RNN は GPU の並列計算能力を十分に活用できません。

In [ ]:
# 逐次処理 vs 並列処理の時間比較（概念図）
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# === 左: RNN の逐次処理 ===
ax = axes[0]
ax.set_xlim(0, 12)
ax.set_ylim(-0.5, 5)
ax.axis('off')
ax.set_title('RNN: 逐次処理（1つずつ順番に）', fontsize=12, fontweight='bold', color='#D32F2F')

words = ['How', 'are', 'you', '?']
for i, word in enumerate(words):
    # 各単語の処理ブロック
    rect = mpatches.FancyBboxPatch((i * 2.5 + 0.5, 2), 2.0, 1.0,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#FFCDD2', edgecolor='#D32F2F', linewidth=2)
    ax.add_patch(rect)
    ax.text(i * 2.5 + 1.5, 2.5, f'[{word}]', ha='center', fontsize=10, fontweight='bold')
    
    # 時間軸の矢印
    if i < len(words) - 1:
        ax.annotate('', xy=(i * 2.5 + 2.7, 2.5), xytext=(i * 2.5 + 2.55, 2.5),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))

# 時間軸
ax.annotate('', xy=(11, 0.5), xytext=(0.5, 0.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.text(5.75, 0.1, '時間', ha='center', fontsize=11, color='gray')
ax.text(5.75, 4.2, '計算時間: O(N) — 単語数に比例', ha='center', fontsize=11,
        color='#D32F2F', fontweight='bold')

# === 右: Transformer の並列処理 ===
ax = axes[1]
ax.set_xlim(0, 12)
ax.set_ylim(-0.5, 5)
ax.axis('off')
ax.set_title('Transformer: 並列処理（同時に処理）', fontsize=12, fontweight='bold', color='#4CAF50')

for i, word in enumerate(words):
    # すべての単語を同時に処理
    rect = mpatches.FancyBboxPatch((1.5, 3.5 - i * 0.9), 8.0, 0.7,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#C8E6C9', edgecolor='#4CAF50', linewidth=2)
    ax.add_patch(rect)
    ax.text(5.5, 3.85 - i * 0.9, f'[{word}] の処理', ha='center', fontsize=10, fontweight='bold')

# 同時処理を示す括弧
ax.annotate('', xy=(0.8, 3.8), xytext=(0.8, 1.1),
            arrowprops=dict(arrowstyle='-', color='#4CAF50', lw=2))
ax.text(0.4, 2.45, '同\n時\n処\n理', ha='center', fontsize=10, color='#4CAF50', fontweight='bold')

# 時間軸
ax.annotate('', xy=(11, 0.5), xytext=(0.5, 0.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.text(5.75, 0.1, '時間', ha='center', fontsize=11, color='gray')
ax.text(5.75, 4.5, '計算時間: O(1) — 単語数に依存しない', ha='center', fontsize=11,
        color='#4CAF50', fontweight='bold')

plt.tight_layout()
plt.show()

print("逐次処理 vs 並列処理:")
print("  RNN: 単語を1つずつ順番に処理 → 時間は単語数に比例 O(N)")
print("  Transformer: 全単語を同時に処理 → 時間は単語数に依存しない O(1)")
print()
print("  → 長い文ほど、Transformer の速度優位が大きくなる")

In [ ]:
# 処理時間の比較をシミュレーション
import time

def simulate_rnn_time(sequence_length, time_per_step=0.001):
    """RNN の逐次処理時間をシミュレート"""
    return sequence_length * time_per_step

def simulate_transformer_time(sequence_length, base_time=0.002, overhead=0.0001):
    """Transformer の並列処理時間をシミュレート
    
    アテンションの計算コストは O(N²) だが、並列化により実時間は大幅に短縮
    """
    return base_time + overhead * np.sqrt(sequence_length)

# 異なる文の長さでの処理時間を比較
lengths = [10, 20, 50, 100, 200, 500]

print("=== 処理時間の比較（シミュレーション） ===")
print(f"{'文の長さ':>10s} {'RNN時間':>12s} {'Transformer時間':>16s} {'速度比':>10s}")
print("-" * 55)

rnn_times = []
transformer_times = []

for length in lengths:
    rnn_t = simulate_rnn_time(length)
    transformer_t = simulate_transformer_time(length)
    speedup = rnn_t / transformer_t
    
    rnn_times.append(rnn_t * 1000)  # ミリ秒に変換
    transformer_times.append(transformer_t * 1000)
    
    print(f"{length:>8d}語 {rnn_t*1000:>10.1f}ms {transformer_t*1000:>14.1f}ms {speedup:>8.1f}x高速")

# グラフで可視化
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(lengths))
width = 0.35

bars1 = ax.bar(x - width/2, rnn_times, width, label='RNN (Seq2seq)', color='#FFCDD2', edgecolor='#D32F2F', linewidth=2)
bars2 = ax.bar(x + width/2, transformer_times, width, label='Transformer', color='#C8E6C9', edgecolor='#4CAF50', linewidth=2)

ax.set_xlabel('文の長さ（単語数）', fontsize=11)
ax.set_ylabel('処理時間 (ms)', fontsize=11)
ax.set_title('文の長さと処理時間の関係（シミュレーション）', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(lengths)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Transformer のアテンション機構による解決

2017年、Google が発表した **Transformer** は、これらの問題をエレガントに解決しました。

### 解決策: アテンション（Attention）機構

**アテンション** = 「どの入力に注目すべきか」を動的に決める仕組み

| Seq2seq の問題 | Transformer の解決策 |
|----------------|---------------------|
| ボトルネック問題 | **全ての隠れ状態を参照**できる |
| 逐次処理の遅さ | **並列処理**が可能 |

### アテンションの直感的な理解

翻訳で「beautiful」を「美しい」に変換するとき...

- **Seq2seq**: 最終隠れ状態だけを見る（「beautiful」の情報が薄まっているかも）
- **Transformer**: 「beautiful」に直接注目できる（重みを高くする）

```
入力: [How] [are] [you] [?]
       ↓     ↓     ↓    ↓
重み: 0.1   0.1   0.7  0.1  ← 「you」に注目！
```

In [ ]:
# Seq2seq vs Transformer のアーキテクチャ比較（書籍の図2-4を再現）
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# === 左: Seq2seq（RNN ベース） ===
ax = axes[0]
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Seq2seq（RNNベース）', fontsize=13, fontweight='bold', color='#1565C0')

# エンコーダ側の単語
enc_words = ['How', 'are', 'you', '?']
for i, word in enumerate(enc_words):
    x = i * 1.8 + 1
    ax.text(x, 6.5, f'[{word}]', ha='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='#E3F2FD', edgecolor='#1565C0'))
    # RNN セル
    rect = mpatches.FancyBboxPatch((x-0.5, 5.2), 1.0, 0.8,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#BBDEFB', edgecolor='#1565C0', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, 5.6, 'RNN', ha='center', fontsize=8, fontweight='bold')
    ax.annotate('', xy=(x, 5.15), xytext=(x, 6.2),
                arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1))
    
    # 隠れ状態の矢印
    if i < len(enc_words) - 1:
        ax.annotate('', xy=(x + 1.3, 5.6), xytext=(x + 0.55, 5.6),
                    arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5))

# 最終隠れ状態（ボトルネック）
bottleneck = mpatches.FancyBboxPatch((3.5, 3.5), 2.5, 1.0,
                                      boxstyle='round,pad=0.1',
                                      facecolor='#FFCDD2', edgecolor='#D32F2F', linewidth=3)
ax.add_patch(bottleneck)
ax.text(4.75, 4.0, '最終隠れ状態\n（ボトルネック）', ha='center', fontsize=8, fontweight='bold', color='#D32F2F')
ax.annotate('', xy=(4.75, 3.45), xytext=(6.4, 5.15),
            arrowprops=dict(arrowstyle='->', color='#D32F2F', lw=2))

# デコーダ側
dec_words = ['Como', 'estas', '?']
for i, word in enumerate(dec_words):
    x = i * 1.8 + 2.5
    rect = mpatches.FancyBboxPatch((x-0.5, 1.8), 1.0, 0.8,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#FFE0B2', edgecolor='#E65100', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, 2.2, 'RNN', ha='center', fontsize=8, fontweight='bold')
    ax.text(x, 1.0, f'[{word}]', ha='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='#FFF3E0', edgecolor='#E65100'))
    ax.annotate('', xy=(x, 1.3), xytext=(x, 1.75),
                arrowprops=dict(arrowstyle='->', color='#E65100', lw=1))
    
    # ボトルネックからの矢印（最初のデコーダへ）
    if i == 0:
        ax.annotate('', xy=(x, 2.65), xytext=(4.75, 3.45),
                    arrowprops=dict(arrowstyle='->', color='#D32F2F', lw=2))
    
    # デコーダ間の矢印
    if i < len(dec_words) - 1:
        ax.annotate('', xy=(x + 1.3, 2.2), xytext=(x + 0.55, 2.2),
                    arrowprops=dict(arrowstyle='->', color='#E65100', lw=1.5))

ax.text(6, 0.3, '問題: 最終隠れ状態だけでは情報不足', ha='center', fontsize=10,
        color='#D32F2F', fontweight='bold')

# === 右: Transformer（アテンション） ===
ax = axes[1]
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Transformer（アテンション機構）', fontsize=13, fontweight='bold', color='#4CAF50')

# エンコーダ側の単語
for i, word in enumerate(enc_words):
    x = i * 1.8 + 1
    ax.text(x, 6.5, f'[{word}]', ha='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='#E8F5E9', edgecolor='#4CAF50'))
    # エンコーダブロック
    rect = mpatches.FancyBboxPatch((x-0.5, 5.2), 1.0, 0.8,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#C8E6C9', edgecolor='#4CAF50', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, 5.6, 'Enc', ha='center', fontsize=8, fontweight='bold')
    ax.annotate('', xy=(x, 5.15), xytext=(x, 6.2),
                arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=1))

# すべての隠れ状態を保持
all_states = mpatches.FancyBboxPatch((0.3, 3.8), 8.0, 0.8,
                                      boxstyle='round,pad=0.1',
                                      facecolor='#C8E6C9', edgecolor='#4CAF50', linewidth=2)
ax.add_patch(all_states)
ax.text(4.3, 4.2, '全ての隠れ状態を保持（h₁, h₂, h₃, h₄）', ha='center', fontsize=9, fontweight='bold', color='#2E7D32')

# 各エンコーダから全体へ
for i in range(len(enc_words)):
    x = i * 1.8 + 1
    ax.annotate('', xy=(4.3, 4.65), xytext=(x, 5.15),
                arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=1))

# デコーダ側
for i, word in enumerate(dec_words):
    x = i * 1.8 + 2.5
    rect = mpatches.FancyBboxPatch((x-0.5, 1.8), 1.0, 0.8,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#FFE0B2', edgecolor='#E65100', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, 2.2, 'Dec', ha='center', fontsize=8, fontweight='bold')
    ax.text(x, 1.0, f'[{word}]', ha='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='#FFF3E0', edgecolor='#E65100'))
    ax.annotate('', xy=(x, 1.3), xytext=(x, 1.75),
                arrowprops=dict(arrowstyle='->', color='#E65100', lw=1))
    
    # 全ての隠れ状態への参照（アテンション）
    ax.annotate('', xy=(x, 2.65), xytext=(4.3, 3.75),
                arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=1.5, linestyle='dashed'))

ax.text(6, 0.3, '解決: 全ての入力に直接アクセス可能', ha='center', fontsize=10,
        color='#4CAF50', fontweight='bold')

plt.tight_layout()
plt.show()

print("図の解説（書籍 図2-4 に対応）:")
print("  左（Seq2seq）: 最終隠れ状態だけがデコーダに渡される → ボトルネック")
print("  右（Transformer）: 全ての隠れ状態をデコーダが参照できる → アテンション")

In [ ]:
# アテンションの重みを可視化

# 翻訳の例: "How are you ?" → "Como estas ?"
source_words = ['How', 'are', 'you', '?']
target_words = ['Como', 'estas', '?']

# アテンションの重み（どの入力単語に注目しているか）
# 各行が出力単語、各列が入力単語への注目度
attention_weights = np.array([
    [0.60, 0.15, 0.15, 0.10],  # Como → How に強く注目
    [0.10, 0.55, 0.25, 0.10],  # estas → are に注目
    [0.05, 0.05, 0.10, 0.80],  # ? → ? に強く注目
])

fig, ax = plt.subplots(figsize=(8, 5))

# ヒートマップ
im = ax.imshow(attention_weights, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)

# 軸ラベル
ax.set_xticks(range(len(source_words)))
ax.set_xticklabels(source_words, fontsize=12)
ax.set_yticks(range(len(target_words)))
ax.set_yticklabels(target_words, fontsize=12)

ax.set_xlabel('入力（英語）', fontsize=12)
ax.set_ylabel('出力（スペイン語）', fontsize=12)
ax.set_title('アテンションの重み：どの入力に注目しているか', fontsize=13, fontweight='bold')

# 数値をセルに表示
for i in range(len(target_words)):
    for j in range(len(source_words)):
        weight = attention_weights[i, j]
        color = 'white' if weight > 0.5 else 'black'
        ax.text(j, i, f'{weight:.2f}', ha='center', va='center', fontsize=11, color=color, fontweight='bold')

# カラーバー
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('注目度', fontsize=11)

plt.tight_layout()
plt.show()

print("アテンションの解釈:")
print("  'Como' を出力するとき → 'How' に 60% 注目")
print("  'estas' を出力するとき → 'are' に 55% 注目")
print("  '?' を出力するとき → '?' に 80% 注目")
print()
print("  → 意味的に対応する単語に自然と注目できている！")

## 6. Q, K, V：アテンションの3つの役者

アテンション機構をより深く理解するために、**Q（Query）**、**K（Key）**、**V（Value）** という3つのベクトルを紹介します。

### 図書館での本探しに例えると...

アテンションを「図書館で本を探す人」に例えると、とても分かりやすくなります：

| ベクトル | 英語 | 役割 | 図書館の例え |
|---------|------|------|-------------|
| **Q（クエリ）** | Query | 「何を探しているか」を表す | 「要約を作成するために情報を探している人」 |
| **K（キー）** | Key | 各トークンの「索引」を表す | 本棚の「背表紙」や「ページ番号」 |
| **V（バリュー）** | Value | 各トークンの「実際の内容」を表す | 本の「中身」や「ページの内容」 |

### アテンションの動作

1. **Query（探している人）** が **Key（背表紙）** を見て、どの本が関連しそうか判断する
2. 関連度が高い本ほど、その **Value（中身）** を多く参照する
3. 最終的に、重要な情報を集約した結果を得る

### 具体例：翻訳タスク

「How are you ?」を「Como estas ?」に翻訳するとき：

- **Q**: デコーダの現在状態（「Como」を出力しようとしている）
- **K**: 入力トークン「How, are, you, ?」それぞれの「索引」
- **V**: 入力トークンそれぞれの「意味内容」

→ Q と K の類似度を計算して、V の重み付け和を取る

In [ ]:
# Q, K, V の働きを可視化（書籍 図2-5 に対応）
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('図2-5: アテンション機構の働き — Q, K, V の役割', fontsize=14, fontweight='bold')

# === 入力トークン（直前のトークン）===
input_tokens = ['How', 'are', 'you', '?', '¿']
for i, token in enumerate(input_tokens):
    x = 2 + i * 1.8
    # トークンボックス
    rect = mpatches.FancyBboxPatch((x-0.6, 7.5), 1.2, 0.8,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, 7.9, token, ha='center', fontsize=11, fontweight='bold')

ax.text(6, 8.8, '直前のトークン（入力）', ha='center', fontsize=12, fontweight='bold', color='#1565C0')

# === キーベクトル（K）===
ax.text(4, 6.3, 'キーベクトル (K)', ha='center', fontsize=10, fontweight='bold', color='#7B1FA2')
for i in range(5):
    x = 2 + i * 1.8
    # K ベクトル
    rect = mpatches.FancyBboxPatch((x-0.4, 5.5), 0.8, 0.6,
                                    boxstyle='round,pad=0.05',
                                    facecolor='#E1BEE7', edgecolor='#7B1FA2', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, 5.8, f'K{i+1}', ha='center', fontsize=9)
    # 矢印
    ax.annotate('', xy=(x, 6.1), xytext=(x, 7.45),
                arrowprops=dict(arrowstyle='->', color='#7B1FA2', lw=1))

# === バリューベクトル（V）===
ax.text(4, 4.5, 'バリューベクトル (V)', ha='center', fontsize=10, fontweight='bold', color='#00796B')
for i in range(5):
    x = 2 + i * 1.8
    # V ベクトル
    rect = mpatches.FancyBboxPatch((x-0.4, 3.7), 0.8, 0.6,
                                    boxstyle='round,pad=0.05',
                                    facecolor='#B2DFDB', edgecolor='#00796B', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, 4.0, f'V{i+1}', ha='center', fontsize=9)
    # 矢印
    ax.annotate('', xy=(x, 4.3), xytext=(x, 5.45),
                arrowprops=dict(arrowstyle='->', color='#00796B', lw=1))

# === クエリベクトル（Q）===
ax.text(12, 6.3, 'クエリベクトル (Q)', ha='center', fontsize=10, fontweight='bold', color='#E65100')
rect = mpatches.FancyBboxPatch((11.3, 5.5), 1.4, 0.6,
                                boxstyle='round,pad=0.05',
                                facecolor='#FFE0B2', edgecolor='#E65100', linewidth=2)
ax.add_patch(rect)
ax.text(12, 5.8, 'Q', ha='center', fontsize=11, fontweight='bold')

# 出力トークン
rect = mpatches.FancyBboxPatch((11.3, 7.5), 1.4, 0.8,
                                boxstyle='round,pad=0.1',
                                facecolor='#FFF3E0', edgecolor='#E65100', linewidth=1.5)
ax.add_patch(rect)
ax.text(12, 7.9, 'Como', ha='center', fontsize=11, fontweight='bold')
ax.text(12, 8.8, '次のトークン', ha='center', fontsize=12, fontweight='bold', color='#E65100')
ax.annotate('', xy=(12, 6.15), xytext=(12, 7.45),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=1.5))

# === Q と K の内積（スコア計算）===
ax.text(8, 5.0, 'Q × K の内積\n（スコア計算）', ha='center', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#FFF9C4', edgecolor='#F9A825'))

# Q から K への矢印
for i in range(5):
    x = 2 + i * 1.8
    ax.annotate('', xy=(x + 0.5, 5.8), xytext=(11.2, 5.8),
                arrowprops=dict(arrowstyle='->', color='#F9A825', lw=1, linestyle='dashed'))

# === V の重み付け和 ===
rect = mpatches.FancyBboxPatch((5.5, 2.0), 3.0, 0.8,
                                boxstyle='round,pad=0.1',
                                facecolor='#C8E6C9', edgecolor='#388E3C', linewidth=2)
ax.add_patch(rect)
ax.text(7, 2.4, '文脈ベクトル', ha='center', fontsize=10, fontweight='bold', color='#388E3C')

# V から文脈ベクトルへの矢印
for i in range(5):
    x = 2 + i * 1.8
    ax.annotate('', xy=(7, 2.85), xytext=(x, 3.65),
                arrowprops=dict(arrowstyle='->', color='#388E3C', lw=1))

ax.text(7, 1.3, 'V₁×w₁ + V₂×w₂ + V₃×w₃ + V₄×w₄ + V₅×w₅', ha='center', fontsize=10,
        color='#388E3C', fontweight='bold')
ax.text(7, 0.7, '（w は Softmax で正規化されたアテンション重み）', ha='center', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

print("Q, K, V の役割まとめ:")
print("  Q（Query）: 「何を探しているか」— デコーダの現在状態")
print("  K（Key）  : 「索引」— 各入力トークンの識別子")
print("  V（Value）: 「中身」— 各入力トークンの実際の情報")
print()
print("  処理の流れ:")
print("    1. Q と各 K の内積でスコア（関連度）を計算")
print("    2. Softmax でスコアを正規化して重み w を得る")
print("    3. V の重み付け和で文脈ベクトルを生成")

## 7. アテンションの計算式

### Q, K, V の生成

入力 $x$ に対して、3つの重み行列 $W_Q$, $W_K$, $W_V$ を掛けて Q, K, V を生成します：

$$Q = x W_Q$$
$$K = x W_K$$
$$V = x W_V$$

### Scaled Dot-Product Attention

アテンションの計算式は以下の通りです：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

| 記号 | 意味 |
|------|------|
| $Q$ | クエリ行列 |
| $K$ | キー行列 |
| $V$ | バリュー行列 |
| $K^T$ | K の転置（行と列を入れ替え） |
| $d_k$ | キーの次元数 |
| $\sqrt{d_k}$ | スケーリング係数（数値の安定化のため） |

### なぜ $\sqrt{d_k}$ で割るのか？

内積の値は次元数が大きくなると大きくなりがちです。
Softmax は入力値の差が大きいと極端な出力（ほぼ0か1）になってしまいます。

$\sqrt{d_k}$ で割ることで、次元数に関わらず適切なスケールに正規化できます。

In [ ]:
# Scaled Dot-Product Attention を numpy で実装
np.random.seed(42)

def scaled_dot_product_attention(Q, K, V):
    """
    Scaled Dot-Product Attention の実装
    
    Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
    """
    d_k = K.shape[-1]  # キーの次元数
    
    # Step 1: Q と K の内積（関連度スコア）
    scores = Q @ K.T
    print(f"Step 1: Q × K^T（スコア行列）")
    print(f"  形状: {scores.shape}")
    print(f"  値:\n{scores.round(2)}")
    
    # Step 2: sqrt(d_k) でスケーリング
    scaled_scores = scores / np.sqrt(d_k)
    print(f"\nStep 2: スケーリング（÷ √{d_k} = ÷ {np.sqrt(d_k):.2f}）")
    print(f"  値:\n{scaled_scores.round(2)}")
    
    # Step 3: Softmax で正規化
    attention_weights = softmax(scaled_scores)
    print(f"\nStep 3: Softmax で正規化（各行の和 = 1）")
    print(f"  アテンション重み:\n{attention_weights.round(3)}")
    
    # Step 4: V との重み付け和
    output = attention_weights @ V
    print(f"\nStep 4: V との重み付け和")
    print(f"  出力:\n{output.round(3)}")
    
    return output, attention_weights

def softmax(x):
    """行ごとに Softmax を適用"""
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

# 簡単な例で計算
print("=== Scaled Dot-Product Attention の計算例 ===")
print()

# 3つのトークン、4次元のベクトル
d_model = 4
n_tokens = 3

# Q, K, V（簡略化のため、ここでは直接定義）
Q = np.array([
    [0.8, 0.2, 0.1, 0.3],  # トークン1のクエリ
    [0.2, 0.9, 0.4, 0.1],  # トークン2のクエリ
    [0.1, 0.3, 0.8, 0.5],  # トークン3のクエリ
])

K = np.array([
    [0.9, 0.1, 0.2, 0.4],  # トークン1のキー
    [0.3, 0.8, 0.5, 0.2],  # トークン2のキー
    [0.2, 0.4, 0.9, 0.6],  # トークン3のキー
])

V = np.array([
    [1.0, 0.0, 0.0, 0.0],  # トークン1のバリュー
    [0.0, 1.0, 0.0, 0.0],  # トークン2のバリュー
    [0.0, 0.0, 1.0, 0.0],  # トークン3のバリュー
])

print(f"Q（クエリ）:\n{Q}")
print(f"\nK（キー）:\n{K}")
print(f"\nV（バリュー）:\n{V}")
print()
print("=" * 50)
print()

output, weights = scaled_dot_product_attention(Q, K, V)

print()
print("=" * 50)
print("解釈:")
print("  各行の出力は、その行のクエリに最も関連する")
print("  バリューの重み付け和になっている")

## 8. Transformer ブロックの構造

Transformer は複数の **Transformer ブロック** を積み重ねた構造をしています。

### 各ブロックの構成

| モジュール | 役割 |
|-----------|------|
| **アテンションモジュール** | Q, K, V, O の4つの重み行列で構成。入力間の関係性を計算 |
| **MLP モジュール** | 2つの全結合層（FF1, FF2）と活性化関数。特徴変換を行う |

### MLP（多層パーセプトロン）モジュール

MLP は **非線形活性化関数** で区切られた線形層で構成されます：

- **ReLU**: $\text{ReLU}(x) = \max(0, x)$（GPT-2 で使用）
- **GELU**: より滑らかな活性化関数（GPT-3 で使用）

### Transformer モデルの主要パラメータ

| パラメータ | 記号 | 説明 | 例（Llama 2-7B） |
|-----------|------|------|-----------------|
| モデル次元 | $d_{model}$ | 隠れ層のサイズ | 4096 |
| ブロック数 | $N$ | Transformer ブロックの数 | 32 |
| FF 次元 | $d_{ff}$ | フィードフォワード層の次元 | 11008 |
| 語彙サイズ | $V$ | 語彙辞書の単語数 | 32000 |

### マルチヘッドアテンション

実際の Transformer では、アテンションを **複数のヘッド** に分割して並列計算します：

- Llama 2-7B: 32 ヘッド、各ヘッドは 128 次元（4096 ÷ 32 = 128）
- 各ヘッドが異なる「視点」で関係性を学習できる

In [ ]:
# Transformer ブロックの構造を可視化（書籍 図2-6 に対応）
fig, ax = plt.subplots(figsize=(16, 6))
ax.set_xlim(0, 16)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('図2-6: Transformer モデルの構造', fontsize=14, fontweight='bold')

# === 入力 ===
ax.text(0.8, 3, '入力', ha='center', fontsize=11, fontweight='bold')
ax.annotate('', xy=(1.4, 3), xytext=(1.1, 3),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# === 埋め込み層 ===
rect = mpatches.FancyBboxPatch((1.5, 2.3), 1.5, 1.4,
                                boxstyle='round,pad=0.1',
                                facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(rect)
ax.text(2.25, 3.2, '埋め込み', ha='center', fontsize=10, fontweight='bold')
ax.text(2.25, 2.7, '+ 位置', ha='center', fontsize=9)

ax.annotate('', xy=(3.2, 3), xytext=(3.05, 3),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# === N 個の Transformer ブロック ===
# ブロック全体の枠
block_rect = mpatches.FancyBboxPatch((3.3, 1.0), 8.5, 4.0,
                                      boxstyle='round,pad=0.2',
                                      facecolor='#FAFAFA', edgecolor='#757575',
                                      linewidth=2, linestyle='dashed')
ax.add_patch(block_rect)
ax.text(7.55, 5.3, 'N 個の Transformer ブロック', ha='center', fontsize=11, fontweight='bold', color='#757575')

# --- アテンションモジュール ---
attn_rect = mpatches.FancyBboxPatch((3.8, 1.5), 3.5, 3.0,
                                     boxstyle='round,pad=0.1',
                                     facecolor='#E8F5E9', edgecolor='#388E3C', linewidth=2)
ax.add_patch(attn_rect)
ax.text(5.55, 4.2, 'アテンション', ha='center', fontsize=11, fontweight='bold', color='#388E3C')

# Q, K, V, O 行列
qkvo_labels = ['Q', 'K', 'V', 'O']
qkvo_colors = ['#FFE0B2', '#E1BEE7', '#B2DFDB', '#FFCCBC']
for i, (label, color) in enumerate(zip(qkvo_labels, qkvo_colors)):
    x = 4.3 + i * 0.8
    rect = mpatches.FancyBboxPatch((x-0.3, 2.8), 0.6, 0.8,
                                    boxstyle='round,pad=0.05',
                                    facecolor=color, edgecolor='gray', linewidth=1)
    ax.add_patch(rect)
    ax.text(x, 3.2, label, ha='center', fontsize=10, fontweight='bold')

ax.text(5.55, 2.0, 'd_model × d_model', ha='center', fontsize=8, color='gray')

ax.annotate('', xy=(7.5, 3), xytext=(7.35, 3),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# --- MLP モジュール ---
mlp_rect = mpatches.FancyBboxPatch((7.8, 1.5), 3.5, 3.0,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2)
ax.add_patch(mlp_rect)
ax.text(9.55, 4.2, 'MLP', ha='center', fontsize=11, fontweight='bold', color='#E65100')

# FF1, FF2
ff_labels = ['FF1', 'FF2']
for i, label in enumerate(ff_labels):
    x = 8.8 + i * 1.5
    rect = mpatches.FancyBboxPatch((x-0.5, 2.8), 1.0, 0.8,
                                    boxstyle='round,pad=0.05',
                                    facecolor='#FFCCBC', edgecolor='#E65100', linewidth=1)
    ax.add_patch(rect)
    ax.text(x, 3.2, label, ha='center', fontsize=10, fontweight='bold')

ax.text(9.55, 2.0, 'd_model ↔ d_ff', ha='center', fontsize=8, color='gray')

ax.annotate('', xy=(11.5, 3), xytext=(11.35, 3),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# === 出力層 ===
rect = mpatches.FancyBboxPatch((12.0, 2.3), 1.5, 1.4,
                                boxstyle='round,pad=0.1',
                                facecolor='#F3E5F5', edgecolor='#7B1FA2', linewidth=2)
ax.add_patch(rect)
ax.text(12.75, 3.2, '出力層', ha='center', fontsize=10, fontweight='bold')
ax.text(12.75, 2.7, '(Linear)', ha='center', fontsize=9)

ax.annotate('', xy=(13.7, 3), xytext=(13.55, 3),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# === 出力 ===
ax.text(14.5, 3, '出力', ha='center', fontsize=11, fontweight='bold')

# 重み行列サイズの凡例
ax.text(3.8, 0.5, '重み行列の次元:', fontsize=9, fontweight='bold')
ax.text(3.8, 0.1, 'Q, K, V, O: d_model × d_model    FF1: d_model × d_ff    FF2: d_ff × d_model', fontsize=8, color='gray')

plt.tight_layout()
plt.show()

# パラメータ数の計算例
print("=== Llama 2-7B のパラメータ例 ===")
d_model = 4096
d_ff = 11008
n_layers = 32
vocab_size = 32000

# アテンションモジュールのパラメータ
attn_params = 4 * d_model * d_model  # Q, K, V, O
print(f"アテンション（1層）: 4 × {d_model} × {d_model} = {attn_params:,} パラメータ")

# MLPモジュールのパラメータ
mlp_params = 2 * d_model * d_ff  # FF1 + FF2
print(f"MLP（1層）: 2 × {d_model} × {d_ff} = {mlp_params:,} パラメータ")

# 全層
total_per_layer = attn_params + mlp_params
print(f"1層合計: {total_per_layer:,} パラメータ")
print(f"全{n_layers}層: {total_per_layer * n_layers:,} パラメータ")

# 埋め込み層
embed_params = vocab_size * d_model
print(f"埋め込み層: {vocab_size} × {d_model} = {embed_params:,} パラメータ")

total = total_per_layer * n_layers + embed_params
print(f"\n概算合計: 約 {total / 1e9:.1f}B パラメータ（実際は約7B）")

## 9. コードで比較：RNN vs アテンション

最後に、RNN とアテンションの計算を numpy で実装して比較してみましょう。

In [ ]:
# RNN vs アテンションの情報アクセスの違いを実装
np.random.seed(42)

# 入力文の各単語の隠れ状態（エンコーダの出力）
# 実際には学習で得られるが、ここでは簡略化のためランダム
hidden_dim = 4
encoder_outputs = {
    'How': np.array([0.8, 0.2, 0.1, 0.3]),
    'are': np.array([0.2, 0.9, 0.4, 0.1]),
    'you': np.array([0.1, 0.3, 0.8, 0.5]),
    '?':   np.array([0.1, 0.1, 0.1, 0.9]),
}

print("=== エンコーダの出力（各単語の隠れ状態） ===")
for word, h in encoder_outputs.items():
    print(f"  [{word}]: {h}")

print()
print("=" * 50)
print("【比較1】Seq2seq: 最終隠れ状態のみを使う")
print("=" * 50)

# Seq2seq では最後の隠れ状態だけを使う
final_hidden = encoder_outputs['?']
print(f"デコーダが受け取る情報: {final_hidden}")
print("→ 'How', 'are', 'you' の情報は直接参照できない！")

print()
print("=" * 50)
print("【比較2】Transformer: 全ての隠れ状態にアテンション")
print("=" * 50)

def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

def attention(query, encoder_outputs):
    """
    簡略化したアテンション計算
    
    query: デコーダの現在状態
    encoder_outputs: エンコーダの全出力
    
    返り値: アテンションで重み付けされた文脈ベクトル
    """
    words = list(encoder_outputs.keys())
    values = np.array(list(encoder_outputs.values()))
    
    # 各キーとのスコア（内積で類似度を計算）
    scores = np.array([np.dot(query, v) for v in values])
    print(f"  スコア（query との内積）: {dict(zip(words, scores.round(2)))}")
    
    # Softmax で正規化
    weights = softmax(scores)
    print(f"  アテンション重み: {dict(zip(words, weights.round(3)))}")
    
    # 重み付き和で文脈ベクトルを計算
    context = np.sum(weights[:, np.newaxis] * values, axis=0)
    return context, weights

# デコーダが "Como" を出力しようとしているとき
# query は "Como" に対応する表現（ここでは "How" に近いベクトル）
query_como = np.array([0.7, 0.3, 0.2, 0.2])

print(f"\nデコーダの query（'Como' を出力中）: {query_como}")
context, weights = attention(query_como, encoder_outputs)
print(f"  文脈ベクトル: {context.round(3)}")
print(f"  → 'How' に最も注目している（重み = {weights[0]:.3f}）")

print()

# デコーダが "estas" を出力しようとしているとき
query_estas = np.array([0.3, 0.8, 0.3, 0.1])

print(f"デコーダの query（'estas' を出力中）: {query_estas}")
context, weights = attention(query_estas, encoder_outputs)
print(f"  文脈ベクトル: {context.round(3)}")
print(f"  → 'are' に最も注目している（重み = {weights[1]:.3f}）")

In [ ]:
# 情報量の比較を可視化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# === 左: Seq2seq が受け取る情報 ===
ax = axes[0]
words = list(encoder_outputs.keys())
seq2seq_access = [0, 0, 0, 1]  # 最後だけアクセス可能

bars = ax.bar(words, seq2seq_access, color=['#EEEEEE', '#EEEEEE', '#EEEEEE', '#FFCDD2'],
              edgecolor=['gray', 'gray', 'gray', '#D32F2F'], linewidth=2)
ax.set_ylabel('アクセス可能', fontsize=11)
ax.set_title('Seq2seq: 最終隠れ状態のみ', fontsize=12, fontweight='bold', color='#D32F2F')
ax.set_ylim(0, 1.2)
ax.set_yticks([0, 1])
ax.set_yticklabels(['No', 'Yes'])

# === 右: Transformer が受け取る情報 ===
ax = axes[1]
transformer_access = [1, 1, 1, 1]  # 全てアクセス可能

bars = ax.bar(words, transformer_access, color='#C8E6C9', edgecolor='#4CAF50', linewidth=2)
ax.set_ylabel('アクセス可能', fontsize=11)
ax.set_title('Transformer: 全ての隠れ状態', fontsize=12, fontweight='bold', color='#4CAF50')
ax.set_ylim(0, 1.2)
ax.set_yticks([0, 1])
ax.set_yticklabels(['No', 'Yes'])

plt.tight_layout()
plt.show()

print("まとめ:")
print("  Seq2seq: デコーダは最終隠れ状態（h₄）しか見えない")
print("  Transformer: デコーダは全ての隠れ状態（h₁, h₂, h₃, h₄）に自由にアクセスできる")

## 10. まとめ

| 項目 | Seq2seq (2014) | Transformer (2017) |
|------|----------------|--------------------|
| **基盤技術** | RNN（再帰型ニューラルネット） | Self-Attention |
| **情報の受け渡し** | 最終隠れ状態のみ | 全ての隠れ状態 |
| **処理方式** | 逐次処理（順番に） | 並列処理（同時に） |
| **長文への対応** | 精度が低下（ボトルネック） | 比較的安定 |
| **学習速度** | 遅い（GPU を活かせない） | 速い（GPU を活かせる） |

### Seq2seq の2つの問題点

1. **ボトルネック問題**: 長い文の情報を1つの固定長ベクトルに押し込める必要がある
2. **逐次処理の遅さ**: 並列処理ができず、GPU の計算能力を活かせない

### Transformer の解決策

**アテンション機構** により、デコーダが入力の全ての位置に直接アクセスできるようになった。

- **Q（Query）**：「何を探しているか」
- **K（Key）**：「索引・ラベル」
- **V（Value）**：「実際の内容」

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

→ ボトルネックが解消され、並列処理も可能に！

## 次のステップ

次のノートブック `01_transformer_overview.ipynb` では、Transformer の全体構造を詳しく見ていきます。

- エンコーダとデコーダの内部構造
- 自己回帰的生成の仕組み
- Softmax による確率予測